# Adım 2 — Kafka Producer ile Streaming Veri Üretimi

Bu notebook, `producer/producer.py` uygulamasını **doğrular ve örnek mesajları gözlemler**.

## PDF Adım 2 Gereksinimleri

- Producer, CSV dosyasından satırları okuyarak Kafka topic'ine **JSON** formatında mesaj göndermeli
- Her mesajda **timestamp**, **kullanıcı ID**, **olay tipi**, **ilgili ID** bulunmalı
- Mesaj gönderim hızı **ayarlanabilir** olmalı (örn. 10–100 msg/s)
- Producer logları ile kaç mesaj gönderildiği takip edilebilmeli

## Mesaj Şeması (Producer Çıktısı)

```json
{
  "timestamp":     "2014-08-12T15:42:00+00:00",
  "user_id":       "u_a3f2b9c1e4d6",
  "event_type":    "steam_review",
  "app_id":        730,
  "app_name":      "Counter-Strike: Global Offensive",
  "review_text":   "This game is amazing...",
  "review_score":  1,
  "review_votes":  42
}
```

| PDF Alanı | Mesaj Alanı | Açıklama |
|-----------|-------------|----------|
| timestamp | `timestamp` | Yorumun yazıldığı UTC zaman damgası |
| kullanıcı ID | `user_id` | (app_id + review_text) hash'inden türetilmiş deterministik ID |
| olay tipi | `event_type` | Sabit: `steam_review` |
| ilgili ID | `app_id` | Steam oyun ID'si |

## BÖLÜM 1 — Producer Konfigürasyonu

Producer aşağıdaki environment variable'lar ile ayarlanabilir (`docker-compose.yml`'de tanımlı):

In [ ]:
import subprocess

def run(cmd: str) -> str:
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return ((r.stdout or "") + (r.stderr or "")).strip()

# Mevcut producer environment'ı listele
print(run("docker compose config | grep -A 8 'producer:' | head -30"))

## BÖLÜM 2 — Producer'ı Başlat (henüz çalışmıyorsa)

In [ ]:
# Producer container'ını başlat (zaten çalışıyorsa noop)
print(run("docker compose up -d producer"))
print()

# Producer durumunu kontrol et
print(run("docker compose ps producer"))

## BÖLÜM 3 — Producer Loglarından Gönderim İstatistiklerini İzle

In [ ]:
# Son 50 satır log
print(run("docker compose logs --tail=50 producer"))

## BÖLÜM 4 — Kafka'ya Gerçekten Mesaj Düşüyor mu? (Console Consumer)

Aşağıdaki hücre **5 saniye boyunca** Kafka'dan mesaj okur ve örnekleri yazdırır.

In [ ]:
import json

# 5 saniye içinde gelen ilk 5 mesajı çek
cmd = (
    "docker exec kafka timeout 5 "
    "kafka-console-consumer "
    "--bootstrap-server localhost:9092 "
    "--topic steam-reviews "
    "--from-beginning "
    "--max-messages 5"
)
raw = run(cmd)
print("=== Ham çıktı ===")
print(raw)
print("\n=== JSON parse edilmiş örnek mesajlar ===")
for line in raw.splitlines():
    line = line.strip()
    if not line.startswith("{"):
        continue
    try:
        msg = json.loads(line)
        print(json.dumps(msg, indent=2, ensure_ascii=False))
        print("-" * 50)
    except json.JSONDecodeError:
        pass

## BÖLÜM 5 — Mesaj Şeması Doğrulama (PDF Gereksinimi)

In [ ]:
REQUIRED_FIELDS = {
    "timestamp":   "PDF: timestamp",
    "user_id":     "PDF: kullanıcı ID",
    "event_type":  "PDF: olay tipi",
    "app_id":      "PDF: ilgili ID",
}

# 1 mesaj çek
cmd = (
    "docker exec kafka timeout 5 "
    "kafka-console-consumer --bootstrap-server localhost:9092 "
    "--topic steam-reviews --from-beginning --max-messages 1"
)
raw = run(cmd)

msg = None
for line in raw.splitlines():
    if line.strip().startswith("{"):
        msg = json.loads(line.strip())
        break

if msg is None:
    print("⚠ Henüz mesaj alınamadı, producer'ın yeterli veri göndermesini bekleyin.")
else:
    print("Örnek mesaj:")
    print(json.dumps(msg, indent=2, ensure_ascii=False))
    print("\nPDF zorunlu alan kontrolü:")
    for field, label in REQUIRED_FIELDS.items():
        present = field in msg
        mark = "✅" if present else "❌"
        value = msg.get(field, "<eksik>")
        print(f"  {mark} {field:12s} ({label}) = {value}")

## BÖLÜM 6 — Throughput Ölçümü

In [ ]:
import time

def topic_offset_total(topic: str = "steam-reviews") -> int:
    """Topic'in toplam mesaj sayısını GetOffsetShell ile döndürür."""
    out = run(
        f"docker exec kafka kafka-run-class kafka.tools.GetOffsetShell "
        f"--broker-list localhost:9092 --topic {topic} --time -1"
    )
    total = 0
    for line in out.splitlines():
        # format: topic:partition:offset
        parts = line.strip().split(":")
        if len(parts) == 3 and parts[2].isdigit():
            total += int(parts[2])
    return total

before = topic_offset_total()
print(f"t=0  → toplam mesaj: {before:,}")

WINDOW = 10  # saniye
time.sleep(WINDOW)

after = topic_offset_total()
delta = after - before
print(f"t={WINDOW} → toplam mesaj: {after:,}")
print(f"\n📊 Son {WINDOW}s içinde gönderilen: {delta:,} mesaj")
print(f"📊 Throughput: {delta / WINDOW:.1f} msg/s")
print(f"📊 Hedef (MESSAGES_PER_SECOND): 50 msg/s")

## BÖLÜM 7 — Hız Ayarı Değiştirme (PDF: ayarlanabilir gönderim hızı)

```bash
# Producer'ı 100 msg/s ile yeniden başlat
MESSAGES_PER_SECOND=100 docker compose up -d producer --force-recreate

# veya 10 msg/s ile
MESSAGES_PER_SECOND=10 docker compose up -d producer --force-recreate
```

## BÖLÜM 8 — Topic Detayları

In [ ]:
print(run(
    "docker exec kafka kafka-topics --bootstrap-server localhost:9092 "
    "--describe --topic steam-reviews"
))